In [1]:
import pandas as pd
import numpy as np
import math

df = pd.read_csv("../data/processed/search_interactions_features.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 50000
Columns: 19


In [2]:
normal = df.loc[df["is_suspicious"] == 0, "dwell_time_seconds"].values
suspicious = df.loc[df["is_suspicious"] == 1, "dwell_time_seconds"].values

all_values = np.concatenate([normal, suspicious])

# Rank all observations
order = np.argsort(all_values)
ranks = np.empty(len(all_values), dtype=float)
ranks[order] = np.arange(1, len(all_values) + 1)

# Handle tied values
unique_values, inverse, counts = np.unique(
    all_values, return_inverse=True, return_counts=True
)

for i, count in enumerate(counts):
    if count > 1:
        idx = np.where(inverse == i)[0]
        ranks[idx] = ranks[idx].mean()

n1 = len(normal)
n2 = len(suspicious)

U1 = ranks[:n1].sum() - n1 * (n1 + 1) / 2
U2 = n1 * n2 - U1

U = min(U1, U2)

N = n1 + n2
tie_term = np.sum(counts**3 - counts)

variance = (
    n1 * n2 / 12
    * ((N + 1) - tie_term / (N * (N - 1)))
)

z = (U - (n1 * n2 / 2) + 0.5) / math.sqrt(variance)

p_value = math.erfc(abs(z) / math.sqrt(2))

print("Normal events:", n1)
print("Suspicious events:", n2)
print("Mann-Whitney U:", round(U, 2))
print("Z-score:", round(z, 4))
print("Approx. p-value:", p_value)

Normal events: 46078
Suspicious events: 3922
Mann-Whitney U: 6245534.5
Z-score: -96.9317
Approx. p-value: 0.0


In [3]:
table = pd.crosstab(
    df["is_suspicious"],
    df["query_reformulated"]
)

print("Observed Contingency Table:")
print(table)

# Convert to NumPy array
observed = table.values

# Calculate expected frequencies
row_totals = observed.sum(axis=1, keepdims=True)
col_totals = observed.sum(axis=0, keepdims=True)
total = observed.sum()

expected = (row_totals @ col_totals) / total

print("\nExpected Frequencies:")
print(expected)

# Chi-square statistic
chi2 = ((observed - expected) ** 2 / expected).sum()

# For a 2x2 table, degrees of freedom = 1
df_chi = (observed.shape[0] - 1) * (observed.shape[1] - 1)

# Approximate p-value for df = 1
p_value = math.erfc(math.sqrt(chi2 / 2))

print("\nChi-square statistic:", round(chi2, 4))
print("Degrees of freedom:", df_chi)
print("Approximate p-value:", p_value)

# Interpretation
if p_value < 0.05:
    print("\nResult: Statistically significant association.")
else:
    print("\nResult: No statistically significant association.")

Observed Contingency Table:
query_reformulated      0     1
is_suspicious                  
0                   37850  8228
1                    1444  2478

Expected Frequencies:
[[36211.77864  9866.22136]
 [ 3082.22136   839.77864]]

Chi-square statistic: 4412.6603
Degrees of freedom: 1
Approximate p-value: 0.0

Result: Statistically significant association.
